## Conversational Chat Bot

- To make a Question Answer RAG application coversational
- We need to maintian chat history, here is where different types of messages come in handy

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda, RunnableParallel
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

from dotenv import load_dotenv
from typing import List
from pydantic import BaseModel, Field
import os

In [2]:
load_dotenv()
llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0
)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [3]:
def load_multiple_documents(directory_path: str) -> List[Document]:

    documents = []

    for file in os.listdir(directory_path):
        if file.endswith(".docx"):
            loader = Docx2txtLoader(os.path.join(directory_path, file))
        elif file.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(directory_path, file))
        else:
            print(f"Skipping {file} as it is not a supported file type")
            continue
        documents.extend(loader.load())

    return documents

In [4]:
def docs_to_text(documents: List[Document]) -> str:
    return "\n\n".join([doc.page_content for doc in documents])

In [5]:
# loading documents
document_path = "docs"
documents = load_multiple_documents(document_path)
print(f"Number of documents: {len(documents)}")

# splitting documents into consumable chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100, length_function=len)
splits = text_splitter.split_documents(documents)
print(f"Number of documents after splitting: {len(splits)}")

# creating embeddings using OpenAI embeddings
doc_embeddings = embeddings.embed_documents([split.page_content for split in splits])
print(f"Embeddings created for {len(doc_embeddings)} documents")

# creating Chroma vector store and storing embeddings in it
collection_name = "docs_collection"
vectorstore = Chroma.from_documents(splits, embeddings, collection_name=collection_name, persist_directory="chroma_db")
print("Document embeddings added to ChromaDB present in ./chroma_db directory")

# creating retriever to get similar documents from vector store
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("Retriever created, to capture top 2 similar documents from vector stores")

# Setting up output parser - String output parser
parser = StrOutputParser()
print("Output parser created - String output parser for human readable output")

Number of documents: 5
Number of documents after splitting: 8
Embeddings created for 8 documents
Document embeddings added to ChromaDB present in ./chroma_db directory
Retriever created, to capture top 2 similar documents from vector stores
Output parser created - String output parser for human readable output


In [6]:
# Setting up prompt template
template = """
Answer the question as truthfully as possible using the provided context,
and if the answer is not contained within the context, say "I don't know".

Context: {context}
Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)
print("Prompt template created")

Prompt template created


In [7]:
# creating RAG chain
rag_parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(docs_to_text),
    "question": RunnablePassthrough()
})

chain = rag_parallel_chain | prompt | llm | parser

print("RAG chain created")

RAG chain created


Using the above setup let us capture the chat history in a list using message types

In [8]:
chat_history = []

query = "When was GreenGrow Innovations founded?"

chat_history.append(HumanMessage(content=query))
result = chain.invoke(query)
chat_history.append(AIMessage(content=result))

print(result)

print("Chat history: ", chat_history)

GreenGrow Innovations was founded in 2010.
Chat history:  [HumanMessage(content='When was GreenGrow Innovations founded?', additional_kwargs={}, response_metadata={}), AIMessage(content='GreenGrow Innovations was founded in 2010.', additional_kwargs={}, response_metadata={})]


Now that we have this chat history we can use this for LLM to get context about next questions asked by the user

In [9]:
contextualized_system_prompt = """
    Given a chat history and the latest user question which may reference context in the chat history,
    reformulate the question to be a standalone question that can be understood without the chat history.

    Do NOT answer the question, just reformulate it or return it as is.
    """
contextualized_question_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualized_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{question}"),
    ]
)

print("Contextualized prompt template created, to capture context from chat history")
print(contextualized_question_prompt)

Contextualized prompt template created, to capture context from chat history
input_variables=['chat_history', 'question'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMess

In [10]:
contextualized_chain = contextualized_question_prompt | llm | parser
query = "Where is it headquartered?"

contextualized_chain.invoke({"question": query, "chat_history": chat_history})

'What is the headquarters location of GreenGrow Innovations?'

Note: this is not the final answer this is to get the contextualized question so that, LLM can understand what is happening.